# 🎬 YouTube Shorts ワンクリック自動生成

---

## ✏️ 毎回やること（セル1だけ変える）

| 設定項目 | 説明 |
|---|---|
| `YOUTUBE_API_KEY` | **初回だけ**入力。Google Cloud Console で取得（**無料**） |
| `CLAUDE_API_KEY` | **初回だけ**入力。console.anthropic.com で取得（有料・`sk-ant-`で始まる） |
| `PEXELS_API_KEY` | **初回だけ**入力。pexels.com/api で取得（**無料**） |
| `THEME` | **毎回**テーマを書き換える |
| `VIDEO_COUNT` | 生成する本数（1〜10） |

あとは「**ランタイム → すべてのセルを実行**」をクリックするだけ！

---

## 💰 Claude APIの費用目安
| 使い方 | 費用 |
|---|---|
| 1回10本生成 | 約5円 |
| 毎日10本 × 30日 | 約160円/月 |

---

## 🎨 画像スタイル
| スタイル名 | 見た目 | 向いているテーマ |
|---|---|---|
| `realistic` | 写真そのまま | 筋トレ・料理・ビジネス |
| `anime` | アニメ風 | 恋愛・感情・エンタメ |
| `manga` | 漫画風（白黒） | 怖い話・歴史・雑学 |
| `illustration` | イラスト風 | 子ども向け・ライフスタイル |

---

## ⏱ 目安時間
| 本数 | 目安 |
|---|---|
| 1本 | 約5〜10分 |
| 5本 | 約25〜40分 |
| 10本 | 約50〜80分 |

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  セル1: ここだけ変える（毎回）                        ║
# ╚══════════════════════════════════════════════════════╝

# ── APIキー ──────────────────────────────────────────────
YOUTUBE_API_KEY = ''   # ← Google Cloud Console で取得（無料）
CLAUDE_API_KEY  = ''   # ← console.anthropic.com で取得（sk-ant-で始まる）
PEXELS_API_KEY  = ''   # ← pexels.com/api で取得（無料）

# ── テーマ（毎回変える） ─────────────────────────────────
THEME = 'ダイエット'
#  例: 'ダイエット' / '筋トレ' / 'NISA' / '投資' / '英語学習' / '副業'

# ── 生成する動画の本数 ───────────────────────────────────
VIDEO_COUNT = 10        # ← 1〜10本

# ── 動画の長さ ───────────────────────────────────────────
DURATION = 45           # ← 秒数（30〜60）

# ── 画像スタイル ─────────────────────────────────────────
IMAGE_STYLE = 'realistic'    # 写真そのまま
# IMAGE_STYLE = 'anime'        # アニメ風
# IMAGE_STYLE = 'manga'        # 漫画風（白黒）
# IMAGE_STYLE = 'illustration' # イラスト風

# ╔══════════════════════════════════════════════════════╗
# ║  ↑ 変えるのはここまで。以下は触らなくてOK            ║
# ╚══════════════════════════════════════════════════════╝

if not YOUTUBE_API_KEY.strip():
    raise ValueError('❌ YOUTUBE_API_KEY を入力してください（Google Cloud Console で取得）')
if not CLAUDE_API_KEY.strip():
    raise ValueError('❌ CLAUDE_API_KEY を入力してください（console.anthropic.com で取得）')
if not PEXELS_API_KEY.strip():
    raise ValueError('❌ PEXELS_API_KEY を入力してください（pexels.com/api で取得）')

import os
os.makedirs('/content/output', exist_ok=True)

print('設定内容を確認します...')
print(f'  テーマ      : {THEME}')
print(f'  生成本数    : {VIDEO_COUNT}本')
print(f'  動画の長さ  : {DURATION}秒')
print(f'  画像スタイル: {IMAGE_STYLE}')
print()
print('✅ 設定完了！')

In [ ]:
# 【自動】必要なツールをインストール（触らなくてOK）
!pip install -q gtts requests opencv-python-headless
!apt-get install -q -y ffmpeg
print('✅ ツールのインストール完了')

In [ ]:
# 【自動】① YouTubeトレンド取得 → ② Claude AI分析 → ③ 切り口生成（触らなくてOK）

import requests as _req
import json, re

# ── YouTube Data API でリアルなトレンドデータを取得 ──────────
def fetch_youtube_trends(theme):
    """YouTubeから実際の人気動画データを取得する"""
    url = 'https://www.googleapis.com/youtube/v3/search'
    params = {
        'part': 'snippet',
        'q': f'{theme} shorts',
        'type': 'video',
        'order': 'viewCount',
        'regionCode': 'JP',
        'relevanceLanguage': 'ja',
        'maxResults': 10,
        'key': YOUTUBE_API_KEY,
        'videoDuration': 'short',
    }
    try:
        r = _req.get(url, params=params, timeout=15)
        if r.status_code == 200:
            items = r.json().get('items', [])
            return [{
                'title': item['snippet'].get('title', ''),
                'description': item['snippet'].get('description', '')[:120],
                'channel': item['snippet'].get('channelTitle', ''),
            } for item in items]
        else:
            print(f'  ⚠ YouTube API エラー: {r.status_code} - {r.text[:100].replace(YOUTUBE_API_KEY, "***")}')
            return []
    except Exception as e:
        print(f'  ⚠ YouTube API 取得失敗: {e}')
        return []

# ── Claude API で台本を生成 ──────────────────────────────────
CLAUDE_URL = 'https://api.anthropic.com/v1/messages'

def call_ai(prompt, tokens=4096):
    res = _req.post(
        CLAUDE_URL,
        headers={
            'x-api-key': CLAUDE_API_KEY,
            'anthropic-version': '2023-06-01',
            'content-type': 'application/json',
        },
        json={
            'model': 'claude-3-5-haiku-20241022',
            'max_tokens': tokens,
            'messages': [{'role': 'user', 'content': prompt}],
        },
        timeout=120
    )
    if res.status_code != 200:
        err = res.text[:300].replace(CLAUDE_API_KEY, '***')
        raise RuntimeError(f'Claude APIエラー ({res.status_code}): {err}')
    return res.json()['content'][0]['text']

# ── ① YouTubeトレンドデータを取得 ───────────────────────────
print(f'📺 YouTubeで「{THEME}」の人気動画を取得中...')
yt_videos = fetch_youtube_trends(THEME)

if yt_videos:
    print(f'  ✓ {len(yt_videos)}件のトレンド動画を取得')
    for i, v in enumerate(yt_videos[:5]):
        print(f'    {i+1}. {v["title"][:45]}')
    yt_data_text = '\n'.join([f'{i+1}. タイトル:「{v["title"]}」' for i, v in enumerate(yt_videos)])
else:
    print('  ⚠ YouTube APIからデータ取得できず。AI知識でトレンド分析します。')
    yt_data_text = f'テーマ「{THEME}」の一般的なトレンド情報を使用'

# ── ② Claude でトレンド分析 ──────────────────────────────────
print(f'\n🤖 Claude AIで「{THEME}」のトレンドを分析中...')
trend = call_ai(
    f'あなたはYouTubeショート動画のトレンドアナリストです。\n'
    f'\n【YouTubeの人気動画データ】\n{yt_data_text}\n\n'
    f'このデータをもとに、テーマ「{THEME}」の{DURATION}秒YouTube Shortsで\n'
    f'バズりやすいフック・構成・差別化のコツを台本制作に直接使える形で簡潔にまとめてください。'
)
print('  ✓ トレンド分析完了')

# ── ③ VIDEO_COUNT個の切り口を生成 ────────────────────────────
print(f'\n💡 「{THEME}」の切り口を{VIDEO_COUNT}個考案中...')
angles_raw = call_ai(
    f'テーマ「{THEME}」のYouTube Shortsで、それぞれ異なる視点・ターゲット・切り口の動画タイトルを{VIDEO_COUNT}個考えてください。\n'
    f'トレンド分析: {trend[:500]}\n\n'
    f'YouTubeの人気動画: {yt_data_text[:300]}\n\n'
    f'出力形式（番号なし、1行1タイトル、日本語のみ）:\n'
    f'タイトル1\nタイトル2\n...\nタイトル{VIDEO_COUNT}',
    tokens=1024
)

ANGLES = [line.strip() for line in angles_raw.strip().split('\n') if line.strip()][:VIDEO_COUNT]
while len(ANGLES) < VIDEO_COUNT:
    ANGLES.append(f'{THEME}の攻略法 Vol.{len(ANGLES)+1}')

print(f'\n生成する{VIDEO_COUNT}本の切り口:')
for i, a in enumerate(ANGLES):
    print(f'  {i+1:2d}. {a}')
print(f'\n✅ 切り口の決定完了')

In [ ]:
# 【自動】③〜⑥ 全動画を一括生成（触らなくてOK）
# 完成したら自動でダウンロードが始まります

import cv2, numpy as np, subprocess, tempfile, time
from pathlib import Path
from datetime import datetime
from gtts import gTTS

def anime(img):
    c = cv2.bilateralFilter(cv2.bilateralFilter(img,9,250,250),9,250,250)
    g = cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),5)
    e = cv2.cvtColor(cv2.adaptiveThreshold(g,255,cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,9,5),cv2.COLOR_GRAY2BGR)
    r = cv2.bitwise_and(c,e)
    h = cv2.cvtColor(r,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1] = np.clip(h[:,:,1]*1.5,0,255)
    return cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)

def manga(img):
    g = cv2.createCLAHE(2.0,(8,8)).apply(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY))
    e = cv2.dilate(cv2.Canny(cv2.GaussianBlur(g,(3,3),0),30,100),np.ones((2,2),np.uint8))
    _,t = cv2.threshold(g,180,255,cv2.THRESH_BINARY)
    return cv2.cvtColor(cv2.addWeighted(t,.75,cv2.bitwise_not(e),.25,0),cv2.COLOR_GRAY2BGR)

def illust(img):
    c = cv2.bilateralFilter(img,15,80,80)
    h = cv2.cvtColor(c,cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:,:,1]=np.clip(h[:,:,1]*1.7,0,255); h[:,:,2]=np.clip(h[:,:,2]*1.1,0,255)
    c = cv2.cvtColor(h.astype(np.uint8),cv2.COLOR_HSV2BGR)
    e = cv2.cvtColor(cv2.adaptiveThreshold(cv2.medianBlur(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY),7),255,
        cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,11,9),cv2.COLOR_GRAY2BGR)
    return cv2.bitwise_and(c,e)

STYLES = {'anime':anime,'manga':manga,'illustration':illust}

def ff(*a):
    r = subprocess.run(['ffmpeg','-y',*[str(x) for x in a]],
                       capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-800:])

def at(s):
    return f'{int(s//3600)}:{int((s%3600)//60):02d}:{int(s%60):02d}.{int((s%1)*100):02d}'

def clean(t):
    t = re.sub(r'\[速く\]|\[ゆっくり\]|\[強調\]','',t)
    t = re.sub(r'\[間\d+\.?\d*\]','、',t)
    t = re.sub(r'（ここにセリフ）|\(ここにセリフ\)','',t)
    t = re.sub(r'シーン\d+[（(][^)）]*[)）]\s*[:：]','',t)
    return re.sub(r'\s+',' ',t).strip()

W, H, FPS = 1080, 1920, 30
SCENE_COUNT = 4 if DURATION <= 35 else 5 if DURATION <= 50 else 6
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)

completed = []
failed = []

def make_clip(img_path, dur, out):
    vf = f"scale={int(W*1.08)}:{int(H*1.08)}:force_original_aspect_ratio=increase,crop={W}:{H}"
    if Path(img_path).exists():
        ff('-loop','1','-i',str(img_path),'-vf',vf,
           '-t',str(dur),'-r',str(FPS),
           '-an','-c:v','libx264','-preset','ultrafast','-crf','23','-pix_fmt','yuv420p',str(out))
    else:
        ff('-f','lavfi','-i',f'color=black:s={W}x{H}:r={FPS}',
           '-t',str(dur),'-c:v','libx264','-preset','ultrafast',str(out))

def make_one_video(idx, angle):
    vid_num = idx + 1
    safe_angle = re.sub(r'[\\/:*?"<>|\s]', '_', angle)[:25]
    print(f'\n{"─"*50}')
    print(f'🎬 [{vid_num}/{VIDEO_COUNT}] {angle}')
    print(f'{"─"*50}')
    TMP = Path(tempfile.mkdtemp(prefix=f'yt{vid_num}_'))
    IMG_TMP = TMP / 'images'
    IMG_TMP.mkdir()

    print('  📝 台本生成中...')
    labels = ['フック','問題提起','本題①','本題②','まとめ','CTA']
    sc_str   = '\n'.join([f'シーン{i+1}（{labels[i]}）:（セリフ）' for i in range(SCENE_COUNT)])
    kw_str   = '\n'.join([f'scene{i+1}:（英語1〜3語）' for i in range(SCENE_COUNT)])
    time_str = '\n'.join([f'scene{i+1}:（秒数）' for i in range(SCENE_COUNT)])
    raw = call_ai(
        f'あなたはYouTubeショート動画の台本専門ライターです。\n'
        f'【タイトル】{angle}\n【テーマ】{THEME}\n【目標尺】{DURATION}秒（{SCENE_COUNT}シーン）\n\n'
        f'以下のフォーマットで出力してください。\n'
        f'[台本]\n{sc_str}\n[キーワード]\n{kw_str}\n[秒数配分]\n{time_str}'
    )

    script_text = (re.search(r'\[台本\]([\s\S]*?)(?=\[キーワード\])', raw) or
                   type('x',(),({'group':lambda s,n:raw}))()).group(1)
    kw_block   = re.search(r'\[キーワード\]([\s\S]*?)(?=\[秒数配分\])', raw)
    time_block = re.search(r'\[秒数配分\]([\s\S]*?)$', raw)

    keywords = []
    if kw_block:
        for line in kw_block.group(1).split('\n'):
            m = re.match(r'scene\d+[:\s]+(.+)', line.strip(), re.I)
            if m: keywords.append(m.group(1).strip())
    while len(keywords) < SCENE_COUNT: keywords.append('lifestyle')

    timings = []
    if time_block:
        for line in time_block.group(1).split('\n'):
            m = re.match(r'scene\d+[:\s]+(\d+)', line.strip(), re.I)
            if m: timings.append(int(m.group(1)))
    while len(timings) < SCENE_COUNT: timings.append(DURATION // SCENE_COUNT)

    scene_lines = re.split(r'シーン\d+[（(][^)）]*[)）]\s*[:：]', script_text)
    scene_texts = [s.strip() for s in scene_lines[1:] if s.strip()]
    while len(scene_texts) < SCENE_COUNT: scene_texts.append(THEME)
    print('  ✓ 台本完了')

    print('  🖼 画像取得中...')
    scenes = []
    for i, (kw, dur) in enumerate(zip(keywords[:SCENE_COUNT], timings[:SCENE_COUNT])):
        time.sleep(0.5)
        fn = IMG_TMP / f'scene_{i+1:02d}.jpg'
        try:
            r = _req.get('https://api.pexels.com/v1/search',
                         headers={'Authorization': PEXELS_API_KEY},
                         params={'query': kw, 'per_page': 1, 'orientation': 'portrait'}, timeout=15)
            if r.status_code == 200:
                photos = r.json().get('photos', [])
                if photos:
                    url = photos[0]['src'].get('portrait') or photos[0]['src'].get('large')
                    dl = _req.get(url, timeout=30)
                    if dl.status_code == 200:
                        fn.write_bytes(dl.content)
        except Exception as e:
            print(f'  ⚠ 画像スキップ(scene{i+1}): {e}')
        scenes.append({'scene': i+1, 'image': fn, 'keyword': kw,
                       'duration': max(dur, 3),
                       'text': scene_texts[i] if i < len(scene_texts) else ''})
    print('  ✓ 画像完了')

    if IMAGE_STYLE in STYLES:
        fn_style = STYLES[IMAGE_STYLE]
        for s in scenes:
            p = s['image']
            if p.exists():
                img = cv2.imread(str(p))
                if img is not None:
                    cv2.imwrite(str(p), fn_style(img))
        print(f'  ✓ スタイル変換完了')

    print('  🎙 音声生成中...')
    wavs = []
    for i, s in enumerate(scenes):
        t = clean(s.get('text', THEME)) or THEME
        mp3, wav = TMP/f'v{i}.mp3', TMP/f'v{i}.wav'
        try:
            gTTS(text=t, lang='ja').save(str(mp3))
            ff('-i',str(mp3),'-ar','44100','-ac','1',str(wav))
            wavs.append(wav)
        except Exception as e:
            print(f'  ⚠ 音声スキップ(scene{i+1}): {e}')
    VOICE = None
    if wavs:
        lf = TMP/'vl.txt'
        lf.write_text('\n'.join(f"file '{p}'" for p in wavs))
        comb, vaac = TMP/'vc.wav', TMP/'v.aac'
        ff('-f','concat','-safe','0','-i',str(lf),'-c','copy',str(comb))
        ff('-i',str(comb),'-c:a','aac','-ar','44100',str(vaac))
        VOICE = vaac
    print('  ✓ 音声完了')

    print('  🎬 動画生成中...')
    clips = []
    for s in scenes:
        out = TMP / f"c{s['scene']:02d}.mp4"
        make_clip(s['image'], s['duration'], out)
        clips.append(out)

    lf2 = TMP/'cl.txt'
    lf2.write_text('\n'.join(f"file '{p}'" for p in clips))
    mg = TMP/'m.mp4'
    ff('-f','concat','-safe','0','-i',str(lf2),'-c','copy',str(mg))

    ah = (f"[Script Info]\nPlayResX:{W}\nPlayResY:{H}\nScriptType:v4.00+\n\n"
          f"[V4+ Styles]\nFormat:Name,Fontname,Fontsize,PrimaryColour,OutlineColour,Bold,Outline,Shadow,Alignment,MarginV\n"
          f"Style:Default,Arial,72,&H00FFFFFF,&H00000000,-1,4,1,2,{int(H*0.18)}\n\n"
          f"[Events]\nFormat:Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text\n")
    ev = []; t = 0.0
    for s in scenes:
        ev.append(f"Dialogue:0,{at(t)},{at(t+s['duration'])},Default,,0,0,0,,{s['keyword'][:20]}")
        t += s['duration']
    sub = TMP/'s.ass'
    sub.write_text(ah+'\n'.join(ev), encoding='utf-8')
    se = str(sub).replace('\\','/').replace(':','\\:')

    safe_theme_f = re.sub(r'[\\/:*?"<>|\s]', '_', THEME)[:15]
    OUT = OUTPUT_DIR / f'{vid_num:02d}_{safe_theme_f}_{safe_angle}.mp4'
    if VOICE and VOICE.exists():
        ff('-i',str(mg),'-i',str(VOICE),'-vf',f'ass={se}','-map','0:v','-map','1:a',
           '-c:v','libx264','-preset','fast','-crf','20','-c:a','aac','-b:a','192k',
           '-pix_fmt','yuv420p','-movflags','+faststart','-t',str(DURATION),str(OUT))
    else:
        ff('-i',str(mg),'-vf',f'ass={se}','-c:v','libx264','-preset','fast','-crf','20',
           '-pix_fmt','yuv420p','-movflags','+faststart','-t',str(DURATION),str(OUT))

    if not OUT.exists() or OUT.stat().st_size < 10000:
        raise RuntimeError(f'動画生成失敗: {OUT}')

    mb = OUT.stat().st_size / 1_048_576
    print(f'  ✅ 完成！ {OUT.name} ({mb:.1f} MB)')
    return str(OUT)

start_all = time.time()
for idx, angle in enumerate(ANGLES):
    try:
        out_path = make_one_video(idx, angle)
        completed.append((idx+1, angle, out_path))
    except Exception as e:
        import traceback
        print(f'\n  ❌ エラー: {traceback.format_exc()[-400:]}')
        failed.append((idx+1, angle, str(e)))

elapsed = (time.time() - start_all) / 60
print(f'\n{"="*50}')
print(f'🎉 生成完了！ 成功:{len(completed)}本 / 失敗:{len(failed)}本 / {elapsed:.1f}分')
print(f'{"="*50}')

In [ ]:
# 【自動】完成動画をダウンロード（触らなくてOK）
from IPython.display import Video, display, Audio
from google.colab import files
import shutil, numpy as np

print(f'✅ 完成した動画 ({len(completed)}本):')
for num, angle, path in completed:
    mb = Path(path).stat().st_size / 1_048_576
    print(f'  {num:2d}. {angle}  [{mb:.1f}MB]')

if failed:
    print(f'\n❌ 失敗 ({len(failed)}本):')
    for num, angle, err in failed:
        print(f'  {num:2d}. {angle} → {err[:80]}')

# プレビュー表示
if completed:
    print('\n▶ 1本目をプレビュー:')
    shutil.copy(completed[0][2], '/content/preview.mp4')
    display(Video('/content/preview.mp4', width=360))

# 完了通知音
sr = 44100
beep = np.sin(2*np.pi*880*np.linspace(0,0.3,int(sr*0.3)))*0.5
silent = np.zeros(int(sr*0.1))
display(Audio(np.concatenate([beep,silent,beep]), rate=sr, autoplay=True))

# ダウンロード
print('\n⬇️ 動画をダウンロードしています...')
for num, angle, path in completed:
    print(f'  ダウンロード中: {Path(path).name}')
    files.download(path)